# 🔀 Conditional Edges

## What are Conditional Edges?
Conditional edges route the graph to different nodes based on **state**.

```
if state has tool_calls → go to tools node
else → go to END
```

This enables:
- Agent loops (keep reasoning until done)
- Error recovery (retry on failure)
- Multi-step workflows (branch based on results)


In [ ]:
# ── Agent with Conditional Edges ─────────────────────────────────────────
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    'Add two integers.'
    return a + b

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

llm = ChatOpenAI(model='gpt-4o-mini').bind_tools([add])

def agent_node(state: AgentState) -> dict:
    response = llm.invoke(state['messages'])
    return {'messages': [response]}

def tool_node(state: AgentState) -> dict:
    from langchain_core.messages import ToolMessage
    last = state['messages'][-1]
    results = []
    for tc in last.tool_calls:
        result = add.invoke(tc['args'])
        results.append(ToolMessage(content=str(result), tool_call_id=tc['id']))
    return {'messages': results}

def should_use_tools(state: AgentState) -> Literal['tools', '__end__']:
    last = state['messages'][-1]
    if hasattr(last, 'tool_calls') and last.tool_calls:
        return 'tools'  # → route to tool_node
    return '__end__'   # → route to END

graph = StateGraph(AgentState)
graph.add_node('agent', agent_node)
graph.add_node('tools', tool_node)
graph.set_entry_point('agent')

# Conditional: agent → tools OR end
graph.add_conditional_edges('agent', should_use_tools)
# After tools: always back to agent (loop!)
graph.add_edge('tools', 'agent')

app = graph.compile()

from langchain_core.messages import HumanMessage
result = app.invoke({'messages': [HumanMessage(content='What is 15 + 27?')]})
print('Final answer:', result['messages'][-1].content)

## 🎯 Interview Questions

1. **What is a conditional edge in LangGraph?**
2. **How do you implement a tool-calling loop with conditional edges?**
3. **What does the routing function return?**
4. **How do you prevent infinite loops in LangGraph?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Conditional Edges — Dynamic Routing**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
